<a href="https://colab.research.google.com/github/MajidFQ/MultiModel-Spam-Classification/blob/main/spam_classifier_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import urllib.request
import zipfile
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# downloading the dataset from UCI repo
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
urllib.request.urlretrieve(url, "sms_spam.zip")

with zipfile.ZipFile("sms_spam.zip", 'r') as z:
    z.extractall("sms_data")

df = pd.read_csv("sms_data/SMSSpamCollection", sep='\t', names=['label_raw', 'text'])

# ham = 0, spam = 1
df['label'] = df['label_raw'].map({'ham': 0, 'spam': 1})

# convert text to numeric features using tfidf
tfidf = TfidfVectorizer(stop_words='english')
X = tfidf.fit_transform(df['text'])
y = df['label']

# 80-20 split, keeping random_state fixed so results dont change every run
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = []

def get_metrics(name, param, y_test, y_pred):
    a = accuracy_score(y_test, y_pred)
    p = precision_score(y_test, y_pred, zero_division=0)
    r = recall_score(y_test, y_pred, zero_division=0)
    f = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    results.append({
        "Model": name,
        "Param": param,
        "Accuracy": round(a, 4),
        "Precision": round(p, 4),
        "Recall": round(r, 4),
        "F1": round(f, 4),
        "Confusion Matrix": f"{cm[0][0]},{cm[0][1]}|{cm[1][0]},{cm[1][1]}"
    })

    print(f"{name} - {param}")
    print(f"acc={a:.4f}  prec={p:.4f}  rec={r:.4f}  f1={f:.4f}")
    print(cm, "\n")


# ---- Naive Bayes with different thresholds ----
nb = MultinomialNB()
nb.fit(X_train, y_train)
probs = nb.predict_proba(X_test)[:, 1]   # prob of spam class

for t in [0.3, 0.5, 0.75]:
    pred = (probs >= t).astype(int)
    get_metrics("Naive Bayes", f"thresh={t}", y_test, pred)


# ---- Decision Tree with different depths ----
for d in [25, 50, 75]:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    pred = dt.predict(X_test)
    get_metrics("Decision Tree", f"depth={d}", y_test, pred)


# ---- SVM with different gamma values ----
for g in [0.01, 0.1, 0.5]:
    svm = SVC(gamma=g, kernel='rbf', random_state=42)
    svm.fit(X_train, y_train)
    pred = svm.predict(X_test)
    get_metrics("SVM", f"gamma={g}", y_test, pred)


# final table
print("Final comparison:")
final_df = pd.DataFrame(results)
print(final_df.to_string(index=False))

Naive Bayes - thresh=0.3
acc=0.9839  prec=0.9580  rec=0.9195  f1=0.9384
[[960   6]
 [ 12 137]] 

Naive Bayes - thresh=0.5
acc=0.9794  prec=1.0000  rec=0.8456  f1=0.9164
[[966   0]
 [ 23 126]] 

Naive Bayes - thresh=0.75
acc=0.9453  prec=1.0000  rec=0.5906  f1=0.7426
[[966   0]
 [ 61  88]] 

Decision Tree - depth=25
acc=0.9650  prec=0.9167  rec=0.8121  f1=0.8612
[[955  11]
 [ 28 121]] 

Decision Tree - depth=50
acc=0.9695  prec=0.9197  rec=0.8456  f1=0.8811
[[955  11]
 [ 23 126]] 

Decision Tree - depth=75
acc=0.9695  prec=0.9197  rec=0.8456  f1=0.8811
[[955  11]
 [ 23 126]] 

SVM - gamma=0.01
acc=0.8664  prec=0.0000  rec=0.0000  f1=0.0000
[[966   0]
 [149   0]] 

SVM - gamma=0.1
acc=0.9561  prec=1.0000  rec=0.6711  f1=0.8032
[[966   0]
 [ 49 100]] 

SVM - gamma=0.5
acc=0.9776  prec=1.0000  rec=0.8322  f1=0.9084
[[966   0]
 [ 25 124]] 

Final comparison:
        Model       Param  Accuracy  Precision  Recall     F1 Confusion Matrix
  Naive Bayes  thresh=0.3    0.9839     0.9580  0.9195 